In [21]:
from pathlib import Path

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# ============================================================
# 1. CAMINHOS
# ============================================================

BASE_DIR = Path("../../").resolve()

CAMINHO_BASE = (
    BASE_DIR
    / "data"
    / "analytical"
    / "base_analitica_v1.parquet"
)

CAMINHO_PIB = (
    BASE_DIR
    / "data"
    / "dados_populacao"
    / "pib_per_capita_municipio.csv"
)

CAMINHO_POPULACAO = (
    BASE_DIR
    / "data"
    / "dados_populacao"
    / "populacao_por_municipio.csv"
)

CAMINHO_ESTRUTURA = (
    BASE_DIR
    / "data"
    / "dados_escolas"
    / "estrutura_escolas_municipio.csv"
)


# ============================================================
# 2. CARREGAMENTO DA BASE ANALÍTICA
# ============================================================

df = pd.read_parquet(CAMINHO_BASE)

print("Base analítica:", df.shape)


# ============================================================
# 3. PADRONIZAÇÃO DO ID DO MUNICÍPIO
# ============================================================

df["id_municipio"] = df["id_municipio"].astype(str)



# ============================================================
# 4. PIB 2023
# ============================================================

df_pib = pd.read_csv(
    CAMINHO_PIB,
    decimal=","
)

df_pib["id_municipio"] = (
    df_pib["id_municipio"]
    .astype(str)
)

df_pib_2023 = df_pib[
    df_pib["ano"] == 2023
].copy()

if df_pib_2023["id_municipio"].duplicated().any():
    raise ValueError(
        "Existem municípios duplicados no PIB de 2023."
    )

df_pib_2023 = df_pib_2023[
    [
        "id_municipio",
        "pib_per_capita"
    ]
]


# ============================================================
# 5. POPULAÇÃO 2023
# ============================================================

df_pop = pd.read_csv(
    CAMINHO_POPULACAO
)

df_pop["id_municipio"] = df_pop["id_municipio"].astype(str)

df_pop["id_municipio"] = (
    pd.to_numeric(df_pop["id_municipio"], errors="coerce")
    .astype("Int64")
    .astype(str)
)

df_pop_2023 = df_pop[
    df_pop["ano"] == 2023
].copy()

if df_pop_2023["id_municipio"].duplicated().any():
    raise ValueError(
        "Existem municípios duplicados na população de 2023."
    )

df_pop_2023 = df_pop_2023[
    [
        "id_municipio",
        "populacao"
    ]
]


# ============================================================
# 6. ESTRUTURA ESCOLAR 2023
# ============================================================

df_estrutura = pd.read_csv(
    CAMINHO_ESTRUTURA
)

df_estrutura["id_municipio"] = (
    df_estrutura["id_municipio"]
    .astype(str)
)

df_estrutura_2023 = df_estrutura[
    df_estrutura["ano"] == 2023
].copy()


# Verifica duplicidade
if df_estrutura_2023["id_municipio"].duplicated().any():
    raise ValueError(
        "Existem municípios duplicados "
        "na estrutura escolar de 2023."
    )


# Seleciona apenas as variáveis que vamos utilizar
colunas_estrutura = [
    "id_municipio",
    "total_escolas",
    "agua_potavel",
    "acesso_energia",
    "esgoto_tratado",
    "tratamento_lixo",
    "acesso_banheiro",
    "biblioteca_sala_leitura",
    "cozinha",
    "laboratorio_informatica",
    "parque_infantil"
]

df_estrutura_2023 = df_estrutura_2023[
    colunas_estrutura
]


# ============================================================
# 7. MERGES
# ============================================================

df_v3 = df.merge(
    df_pib_2023,
    on="id_municipio",
    how="left",
    validate="one_to_one"
)

df_v3 = df_v3.merge(
    df_pop_2023,
    on="id_municipio",
    how="left",
    validate="one_to_one"
)

df_v3 = df_v3.merge(
    df_estrutura_2023,
    on="id_municipio",
    how="left",
    validate="one_to_one"
)


# ============================================================
# 8. COBERTURA DOS DADOS
# ============================================================

print("\nCobertura das variáveis:")

print(
    "PIB:",
    df_v3["pib_per_capita"].notna().mean()
)

print(
    "População:",
    df_v3["populacao"].notna().mean()
)

print(
    "Estrutura escolar:",
    df_v3["total_escolas"].notna().mean()
)


# ============================================================
# 9. DEFINIÇÃO DAS FEATURES E TARGET
# ============================================================

features = [
    # Alfabetização
    "taxa_alfabetizacao_2023",

    # Participação
    "percentual_participacao_2023",

    # Meta
    "meta_alfabetizacao_2024",

    # Socioeconômicas
    "pib_per_capita",
    "populacao",

    # Territorial
    "UF",
    "Capital",

    # Estrutura escolar
    "total_escolas",
    "agua_potavel",
    "acesso_energia",
    "esgoto_tratado",
    "tratamento_lixo",
    "acesso_banheiro",
    "biblioteca_sala_leitura",
    "cozinha",
    "laboratorio_informatica",
    "parque_infantil"
]

target = "taxa_alfabetizacao_2024"


X = df_v3[features]
y = df_v3[target]


# ============================================================
# 10. SEPARAÇÃO TREINO / TESTE
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)


# ============================================================
# 11. IDENTIFICAÇÃO DAS VARIÁVEIS
# ============================================================

variaveis_numericas = [
    "taxa_alfabetizacao_2023",
    "percentual_participacao_2023",
    "meta_alfabetizacao_2024",
    "pib_per_capita",
    "populacao",
    "total_escolas",
    "agua_potavel",
    "acesso_energia",
    "esgoto_tratado",
    "tratamento_lixo",
    "acesso_banheiro",
    "biblioteca_sala_leitura",
    "cozinha",
    "laboratorio_informatica",
    "parque_infantil"
]

variaveis_categoricas = [
    "UF",
    "Capital"
]


# ============================================================
# 12. PREPROCESSAMENTO
# ============================================================

pipeline_numerico = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

pipeline_categorico = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numericas",
            pipeline_numerico,
            variaveis_numericas
        ),
        (
            "categoricas",
            pipeline_categorico,
            variaveis_categoricas
        )
    ]
)


# ============================================================
# 13. MODELO
# ============================================================

modelo = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            LinearRegression()
        )
    ]
)


# ============================================================
# 14. TREINAMENTO
# ============================================================

modelo.fit(
    X_train,
    y_train
)


# ============================================================
# 15. PREDIÇÃO
# ============================================================

y_pred = modelo.predict(X_test)


# ============================================================
# 16. AVALIAÇÃO
# ============================================================

mae = mean_absolute_error(
    y_test,
    y_pred
)

rmse = mean_squared_error(
    y_test,
    y_pred
) ** 0.5

r2 = r2_score(
    y_test,
    y_pred
)


print("\n===== RESULTADOS V3 =====")
print(f"MAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²:   {r2:.2f}")


# ============================================================
# 17. ANÁLISE DOS ERROS
# ============================================================

avaliacao = pd.DataFrame({
    "real": y_test,
    "previsto": y_pred
})

avaliacao["erro"] = (
    avaliacao["real"]
    - avaliacao["previsto"]
)

avaliacao["erro_abs"] = (
    avaliacao["erro"]
    .abs()
)

print("\n===== ERRO =====")

print(
    f"Erro médio: "
    f"{avaliacao['erro'].mean():.2f}"
)

print(
    f"MAE: "
    f"{avaliacao['erro_abs'].mean():.2f}"
)

print("\n===== MAIORES ERROS =====")

print(
    avaliacao
    .sort_values(
        "erro_abs",
        ascending=False
    )
    .head(10)
)

Base analítica: (5352, 11)

Cobertura das variáveis:
PIB: 1.0
População: 1.0
Estrutura escolar: 1.0

===== RESULTADOS V3 =====
MAE:  9.02
RMSE: 11.85
R²:   0.62

===== ERRO =====
Erro médio: -0.44
MAE: 9.02

===== MAIORES ERROS =====
        real   previsto       erro   erro_abs
15    100.00  41.890991  58.109009  58.109009
838    75.47  33.608639  41.861361  41.861361
1084   98.95  58.089636  40.860364  40.860364
351    82.61  43.243747  39.366253  39.366253
5096   45.65  83.953551 -38.303551  38.303551
2273   97.50  60.061527  37.438473  37.438473
544    92.97  55.660869  37.309131  37.309131
4715   39.77  76.102972 -36.332972  36.332972
3475   24.23  59.261164 -35.031164  35.031164
2749   93.94  59.272217  34.667783  34.667783


In [22]:
nomes_features = (
    modelo
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

# Recupera os coeficientes da regressão
coeficientes = (
    modelo
    .named_steps["model"]
    .coef_
)

# Cria DataFrame
pesos = pd.DataFrame({
    "variavel": nomes_features,
    "peso": coeficientes
})

# Importância pelo valor absoluto do coeficiente
pesos["peso_absoluto"] = (
    pesos["peso"].abs()
)

# Ordena pelas maiores magnitudes
pesos = (
    pesos
    .sort_values(
        "peso_absoluto",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\n========== COEFICIENTES — REGRESSÃO LINEAR V3 ==========")

print(
    pesos.to_string(index=False)
)


========== COEFICIENTES — REGRESSÃO LINEAR V3 ==========
                               variavel       peso  peso_absoluto
                     categoricas__UF_AC -15.889331      15.889331
                     categoricas__UF_CE  15.335491      15.335491
                     categoricas__UF_RS -14.457627      14.457627
                     categoricas__UF_GO  13.811293      13.811293
                     categoricas__UF_BA -13.765637      13.765637
                     categoricas__UF_MG  11.544840      11.544840
                     categoricas__UF_SE -11.124930      11.124930
                     categoricas__UF_ES  10.699552      10.699552
                     categoricas__UF_RN  -9.344317       9.344317
     numericas__taxa_alfabetizacao_2023   8.091862       8.091862
                     categoricas__UF_AM  -6.956741       6.956741
                     categoricas__UF_TO  -5.390102       5.390102
                     categoricas__UF_MT   4.970976       4.970976
                  

In [23]:
pesos_numericos = pesos[
    pesos["variavel"].str.startswith("numericas__")
].copy()

print(
    pesos_numericos.to_string(index=False)
)

                               variavel      peso  peso_absoluto
     numericas__taxa_alfabetizacao_2023  8.091862       8.091862
numericas__percentual_participacao_2023  2.428236       2.428236
               numericas__total_escolas -1.520672       1.520672
                   numericas__populacao  0.837579       0.837579
             numericas__tratamento_lixo  0.798576       0.798576
             numericas__acesso_banheiro -0.781529       0.781529
     numericas__meta_alfabetizacao_2024 -0.466756       0.466756
     numericas__biblioteca_sala_leitura  0.457530       0.457530
             numericas__parque_infantil  0.303179       0.303179
              numericas__acesso_energia  0.265672       0.265672
              numericas__esgoto_tratado  0.162218       0.162218
                numericas__agua_potavel -0.142983       0.142983
     numericas__laboratorio_informatica  0.116803       0.116803
                     numericas__cozinha -0.029818       0.029818
              numericas__

In [24]:

from sklearn.inspection import permutation_importance


resultado_linear = permutation_importance(
    modelo,
    X_test,
    y_test,
    scoring="neg_mean_absolute_error",
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)


# ============================================================
# DATAFRAME DE RESULTADOS
# ============================================================

permutation_linear = pd.DataFrame({
    "variavel": X_test.columns,
    "importancia": resultado_linear.importances_mean,
    "desvio_padrao": resultado_linear.importances_std
})


# Ordena da maior para a menor importância
permutation_linear = (
    permutation_linear
    .sort_values(
        "importancia",
        ascending=False
    )
    .reset_index(drop=True)
)


# ============================================================
# RESULTADO COMPLETO
# ============================================================

print(
    "\n========== PERMUTATION IMPORTANCE — REGRESSÃO LINEAR V3 =========="
)

print(
    permutation_linear.to_string(index=False)
)


# ============================================================
# TOP 15
# ============================================================

print(
    "\n========== TOP 15 — REGRESSÃO LINEAR V3 =========="
)

print(
    permutation_linear
    .head(15)
    .to_string(index=False)
)


========== PERMUTATION IMPORTANCE — REGRESSÃO LINEAR V3 ==========
                    variavel  importancia  desvio_padrao
                          UF     4.799190       0.188268
     taxa_alfabetizacao_2023     4.311542       0.153400
percentual_participacao_2023     0.322193       0.058709
               total_escolas     0.140065       0.034173
             tratamento_lixo     0.078620       0.014745
             acesso_banheiro     0.039382       0.021461
     biblioteca_sala_leitura     0.026192       0.015056
             parque_infantil     0.024846       0.007396
              acesso_energia     0.006464       0.005757
     laboratorio_informatica     0.006106       0.003091
              esgoto_tratado     0.002694       0.003482
                agua_potavel     0.001367       0.003812
                     Capital     0.001323       0.001792
                     cozinha     0.000268       0.000682
              pib_per_capita     0.000003       0.000269
                   p

In [18]:
from pathlib import Path

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# ============================================================
# 1. CAMINHOS
# ============================================================

BASE_DIR = Path("../../").resolve()

CAMINHO_BASE = (
    BASE_DIR
    / "data"
    / "analytical"
    / "base_analitica_v1.parquet"
)

CAMINHO_PIB = (
    BASE_DIR
    / "data"
    / "dados_populacao"
    / "pib_per_capita_municipio.csv"
)

CAMINHO_POPULACAO = (
    BASE_DIR
    / "data"
    / "dados_populacao"
    / "populacao_por_municipio.csv"
)

CAMINHO_ESTRUTURA = (
    BASE_DIR
    / "data"
    / "dados_escolas"
    / "estrutura_escolas_municipio.csv"
)


# ============================================================
# 2. CARREGAMENTO DA BASE ANALÍTICA
# ============================================================

df = pd.read_parquet(CAMINHO_BASE)

print("Base analítica:", df.shape)


# ============================================================
# 3. PADRONIZAÇÃO DO ID DO MUNICÍPIO
# ============================================================

df["id_municipio"] = (
    df["id_municipio"]
    .astype(str)
)


# ============================================================
# 4. PIB 2023
# ============================================================

df_pib = pd.read_csv(
    CAMINHO_PIB,
    decimal=","
)

df_pib["id_municipio"] = (
    df_pib["id_municipio"]
    .astype(str)
)

df_pib_2023 = df_pib[
    df_pib["ano"] == 2023
].copy()

if df_pib_2023["id_municipio"].duplicated().any():
    raise ValueError(
        "Existem municípios duplicados no PIB de 2023."
    )

df_pib_2023 = df_pib_2023[
    [
        "id_municipio",
        "pib_per_capita"
    ]
]


# ============================================================
# 5. POPULAÇÃO 2023
# ============================================================

df_pop = pd.read_csv(
    CAMINHO_POPULACAO
)

df_pop["id_municipio"] = df_pop["id_municipio"].astype(str)

df_pop["id_municipio"] = (
    pd.to_numeric(df_pop["id_municipio"], errors="coerce")
    .astype("Int64")
    .astype(str)
)

df_pop_2023 = df_pop[
    df_pop["ano"] == 2023
].copy()

if df_pop_2023["id_municipio"].duplicated().any():
    raise ValueError(
        "Existem municípios duplicados na população de 2023."
    )

df_pop_2023 = df_pop_2023[
    [
        "id_municipio",
        "populacao"
    ]
]


# ============================================================
# 6. ESTRUTURA ESCOLAR 2023
# ============================================================

df_estrutura = pd.read_csv(
    CAMINHO_ESTRUTURA
)

df_estrutura["id_municipio"] = (
    pd.to_numeric(
        df_estrutura["id_municipio"],
        errors="coerce"
    )
    .astype("Int64")
    .astype(str)
)

df_estrutura_2023 = df_estrutura[
    df_estrutura["ano"] == 2023
].copy()


# Verifica duplicidade
if df_estrutura_2023["id_municipio"].duplicated().any():
    raise ValueError(
        "Existem municípios duplicados "
        "na estrutura escolar de 2023."
    )


# Seleciona as variáveis de estrutura escolar
colunas_estrutura = [
    "id_municipio",
    "total_escolas",
    "agua_potavel",
    "acesso_energia",
    "esgoto_tratado",
    "tratamento_lixo",
    "acesso_banheiro",
    "biblioteca_sala_leitura",
    "cozinha",
    "laboratorio_informatica",
    "parque_infantil"
]

df_estrutura_2023 = df_estrutura_2023[
    colunas_estrutura
]


# ============================================================
# 7. MERGE PIB
# ============================================================

df_v3 = df.merge(
    df_pib_2023,
    on="id_municipio",
    how="left",
    validate="one_to_one"
)


# ============================================================
# 8. MERGE POPULAÇÃO
# ============================================================

df_v3 = df_v3.merge(
    df_pop_2023,
    on="id_municipio",
    how="left",
    validate="one_to_one"
)


# ============================================================
# 9. MERGE ESTRUTURA ESCOLAR
# ============================================================

df_v3 = df_v3.merge(
    df_estrutura_2023,
    on="id_municipio",
    how="left",
    validate="one_to_one"
)


# ============================================================
# 10. COBERTURA DOS DADOS
# ============================================================

print("\n========== COBERTURA DOS DADOS ==========")

print(
    "Municípios na base:",
    len(df_v3)
)

print(
    "PIB ausente:",
    df_v3["pib_per_capita"].isna().sum()
)

print(
    "População ausente:",
    df_v3["populacao"].isna().sum()
)

print(
    "Estrutura escolar ausente:",
    df_v3["total_escolas"].isna().sum()
)

print(
    f"Cobertura PIB: "
    f"{df_v3['pib_per_capita'].notna().mean():.2%}"
)

print(
    f"Cobertura população: "
    f"{df_v3['populacao'].notna().mean():.2%}"
)

print(
    f"Cobertura estrutura escolar: "
    f"{df_v3['total_escolas'].notna().mean():.2%}"
)


# ============================================================
# 11. UNIVERSO DO MODELO
# ============================================================

df_modelo = df_v3[
    df_v3["meta_alfabetizacao_2024"].notna()
].copy()

print("\n========== UNIVERSO DO MODELO ==========")

print(
    "Municípios utilizados:",
    len(df_modelo)
)


# ============================================================
# 12. FEATURES E TARGET
# ============================================================

features = [
    # Alfabetização
    "taxa_alfabetizacao_2023",

    # Participação
    "percentual_participacao_2023",

    # Meta
    "meta_alfabetizacao_2024",

    # Socioeconômicas
    "pib_per_capita",
    "populacao",

    # Territoriais
    "UF",
    "Capital",

    # Estrutura escolar
    "total_escolas",
    "agua_potavel",
    "acesso_energia",
    "esgoto_tratado",
    "tratamento_lixo",
    "acesso_banheiro",
    "biblioteca_sala_leitura",
    "cozinha",
    "laboratorio_informatica",
    "parque_infantil"
]

target = "taxa_alfabetizacao_2024"

X = df_modelo[features]
y = df_modelo[target]


# ============================================================
# 13. TRAIN / TEST
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)


# ============================================================
# 14. VARIÁVEIS NUMÉRICAS E CATEGÓRICAS
# ============================================================

variaveis_numericas = [
    "taxa_alfabetizacao_2023",
    "percentual_participacao_2023",
    "meta_alfabetizacao_2024",
    "pib_per_capita",
    "populacao",
    "total_escolas",
    "agua_potavel",
    "acesso_energia",
    "esgoto_tratado",
    "tratamento_lixo",
    "acesso_banheiro",
    "biblioteca_sala_leitura",
    "cozinha",
    "laboratorio_informatica",
    "parque_infantil"
]

variaveis_categoricas = [
    "UF",
    "Capital"
]


# ============================================================
# 15. PREPROCESSAMENTO
# ============================================================

pipeline_numerico = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ]
)

pipeline_categorico = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numericas",
            pipeline_numerico,
            variaveis_numericas
        ),
        (
            "categoricas",
            pipeline_categorico,
            variaveis_categoricas
        )
    ]
)


# ============================================================
# 16. RANDOM FOREST
# ============================================================

modelo = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            RandomForestRegressor(
                n_estimators=300,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)


# ============================================================
# 17. TREINAMENTO
# ============================================================

modelo.fit(
    X_train,
    y_train
)


# ============================================================
# 18. PREDIÇÃO
# ============================================================

y_pred = modelo.predict(X_test)


# ============================================================
# 19. AVALIAÇÃO
# ============================================================

mae = mean_absolute_error(
    y_test,
    y_pred
)

rmse = mean_squared_error(
    y_test,
    y_pred
) ** 0.5

r2 = r2_score(
    y_test,
    y_pred
)


print("\n========== RANDOM FOREST V3 ==========")

print(
    f"MAE:  {mae:.2f}"
)

print(
    f"RMSE: {rmse:.2f}"
)

print(
    f"R²:   {r2:.2f}"
)


# ============================================================
# 20. ANÁLISE DOS ERROS
# ============================================================

avaliacao = pd.DataFrame({
    "real": y_test,
    "previsto": y_pred
})

avaliacao["erro"] = (
    avaliacao["real"]
    - avaliacao["previsto"]
)

avaliacao["erro_absoluto"] = (
    avaliacao["erro"]
    .abs()
)


print("\n========== ERROS ==========")

print(
    f"Erro médio: "
    f"{avaliacao['erro'].mean():.2f}"
)

print(
    f"Erro absoluto médio: "
    f"{avaliacao['erro_absoluto'].mean():.2f}"
)


print("\n========== 10 MAIORES ERROS ==========")

print(
    avaliacao
    .sort_values(
        "erro_absoluto",
        ascending=False
    )
    .head(10)
)

Base analítica: (5352, 11)

========== COBERTURA DOS DADOS ==========
Municípios na base: 5352
PIB ausente: 0
População ausente: 0
Estrutura escolar ausente: 0
Cobertura PIB: 100.00%
Cobertura população: 100.00%
Cobertura estrutura escolar: 100.00%

========== UNIVERSO DO MODELO ==========
Municípios utilizados: 5232

========== RANDOM FOREST V3 ==========
MAE:  8.53
RMSE: 11.29
R²:   0.64

========== ERROS ==========
Erro médio: 0.06
Erro absoluto médio: 8.53

========== 10 MAIORES ERROS ==========
       real   previsto       erro  erro_absoluto
2389   7.10  50.423467 -43.323467      43.323467
3010  24.00  66.190000 -42.190000      42.190000
3425  30.86  72.701033 -41.841033      41.841033
4998  25.00  63.288667 -38.288667      38.288667
3650  38.33  75.406767 -37.076767      37.076767
1694  95.35  59.220933  36.129067      36.129067
2763  95.88  59.881833  35.998167      35.998167
5331  36.34  72.060900 -35.720900      35.720900
3013  89.36  53.890933  35.469067      35.469067
3028 

In [19]:
nomes_features = (
    modelo
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

# Recupera as importâncias da Random Forest
importancias = (
    modelo
    .named_steps["model"]
    .feature_importances_
)

# Cria DataFrame
feature_importance = pd.DataFrame({
    "variavel": nomes_features,
    "importancia": importancias
})

# Ordena da maior para a menor
feature_importance = (
    feature_importance
    .sort_values(
        "importancia",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\n========== FEATURE IMPORTANCE — RF V3 ==========")

print(
    feature_importance.to_string(index=False)
)


========== FEATURE IMPORTANCE — RF V3 ==========
                               variavel  importancia
     numericas__meta_alfabetizacao_2024 3.073657e-01
     numericas__taxa_alfabetizacao_2023 1.397958e-01
                     categoricas__UF_RS 6.944484e-02
numericas__percentual_participacao_2023 6.008226e-02
                   numericas__populacao 5.549301e-02
                     categoricas__UF_MG 5.314093e-02
              numericas__pib_per_capita 4.544243e-02
                     categoricas__UF_BA 4.168901e-02
               numericas__total_escolas 3.236522e-02
     numericas__biblioteca_sala_leitura 2.861107e-02
             numericas__tratamento_lixo 2.402735e-02
             numericas__parque_infantil 2.386993e-02
     numericas__laboratorio_informatica 2.093635e-02
              numericas__esgoto_tratado 1.122012e-02
                numericas__agua_potavel 9.807359e-03
                     categoricas__UF_CE 8.467116e-03
                     categoricas__UF_PB 8.338880e

In [20]:
from sklearn.inspection import permutation_importance


resultado_perm_rf = permutation_importance(
    modelo,
    X_test,
    y_test,
    scoring="neg_mean_absolute_error",
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

# Criar DataFrame
permutation_rf = pd.DataFrame({
    "variavel": X_test.columns,
    "importancia": resultado_perm_rf.importances_mean,
    "desvio_padrao": resultado_perm_rf.importances_std
})

# Ordenar pela importância
permutation_rf = (
    permutation_rf
    .sort_values(
        "importancia",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\n========== TOP 15 — RF V3 ==========")

print(
    permutation_rf
    .head(15)
    .to_string(index=False)
)


========== TOP 15 — RF V3 ==========
                    variavel  importancia  desvio_padrao
     meta_alfabetizacao_2024     3.579084       0.134495
                          UF     3.321364       0.175564
     taxa_alfabetizacao_2023     1.202365       0.062412
percentual_participacao_2023     0.549320       0.076813
                   populacao     0.117894       0.042312
             tratamento_lixo     0.105110       0.038275
              pib_per_capita     0.100268       0.036199
               total_escolas     0.084747       0.037690
     biblioteca_sala_leitura     0.051052       0.022581
                     cozinha     0.019083       0.009349
              esgoto_tratado     0.016572       0.010315
             acesso_banheiro     0.000111       0.008831
                     Capital    -0.000199       0.000023
              acesso_energia    -0.000792       0.007085
     laboratorio_informatica    -0.012377       0.027447
